# TCC WBE-IoT — Challenger XGBoost

Compara o Poisson congelado com XGBoost usando as mesmas features e folds temporais. **O teste de 2024 não é aberto.** A regra de substituição já está registrada em `docs/metodologia/XGBOOST_CHALLENGER.md`.

## 1. Instalar dependências

In [ ]:
%pip -q install "pandas==2.2.3" "numpy==2.1.3" "pyarrow==20.0.0" "scikit-learn==1.6.1" "matplotlib>=3.8,<4" "xgboost==3.0.5"

## 2. Clonar ou atualizar a branch

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git"
GIT_REF = "feature/pipeline-modelagem"
REPO_DIR = Path("/content/tcc-vigilancia-wbe-iot")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{GIT_REF}"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())

## 3. Montar o Drive e localizar o snapshot congelado

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = Path("/content/drive/MyDrive/TCC/datasets/dataset_modelagem_v1_20260927_20261006T182348Z.parquet")
RUNS_DIR = Path("/content/drive/MyDrive/TCC/xgboost_challenger")
EXPECTED_SHA256 = "1d51c54f1d13e28b9253b6db4fa12d9538ed774da6c5d5c33b52f2cd026fcb85"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.exists(), f"Dataset não encontrado: {DATASET_PATH}"

## 4. Validar a identidade do dataset

In [ ]:
import hashlib

digest = hashlib.sha256()
with DATASET_PATH.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f"SHA-256 divergente: {actual_sha256}"
print("Dataset validado:", actual_sha256)

## 5. Executar o challenger

A execução pode levar alguns minutos. Nenhuma rotina deste notebook acessa o teste de 2024.

In [ ]:
import os
import sys

before = {path for path in RUNS_DIR.iterdir() if path.is_dir()}
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR)
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts/run_xgboost_challenger.py"), str(DATASET_PATH), "--output", str(RUNS_DIR)],
    cwd=REPO_DIR, env=env, check=True,
)
after = {path for path in RUNS_DIR.iterdir() if path.is_dir()}
new_runs = sorted(after - before)
assert len(new_runs) == 1, f"Não foi possível identificar a nova execução: {new_runs}"
RUN_DIR = new_runs[0]
print("Execução salva em:", RUN_DIR)

## 6. Conferir decisão e gráfico

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

display(Markdown((RUN_DIR / "RESUMO_EXECUCAO.md").read_text(encoding="utf-8")))
display(pd.read_csv(RUN_DIR / "05_decisao_challenger.csv"))
display(Image(filename=str(RUN_DIR / "07_poisson_vs_xgboost_2023.png")))

## 7. Compactar resultados para análise

In [ ]:
import shutil

zip_path = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("ZIP criado:", zip_path)
print("Envie este ZIP para análise. O teste de 2024 continua bloqueado.")